In [10]:
import pandas as pd
import numpy as np


import os, wrds

os.environ["WRDS_USERNAME"] = "YOUR_WRDS_USERNAME"

db = wrds.Connection(wrds_username = os.environ["WRDS_USERNAME"])
print(" Successful")

Loading library list...
Done
 Successful


In [ ]:
# 배당성과 수익성의 관계 분석

answer = db.raw_sql("""

                    WITH div_y AS(
                    SELECT 
                        d.permno,
                        EXTRACT(YEAR FROM d.dclrdt) AS year,
                        SUM(d.divamt) AS total_div,
                        ABS(AVG(d.divamt/NULLIF(px.prc,0)))*100 AS avg_div
                    FROM crsp.dse d
                    LEFT JOIN LATERAL(
                        SELECT f.date, f.prc
                        FROM crsp.dsf f
                        WHERE d.permno = f.permno
                        AND f.date <= d.dclrdt
                        ORDER BY f.date DESC
                        LIMIT 1) px ON TRUE
                    WHERE d.permno = 14593
                    GROUP BY (d.permno, EXTRACT(YEAR FROM d.dclrdt))
                    ),
                    -- permno, 년도, 배당금, 배당률


                    fund AS(
                    SELECT 
                        f.gvkey,
                        EXTRACT(YEAR FROM f.datadate) AS year,
                        SUM(f.niq) AS total_niq,
                        AVG(f.atq) AS total_asset
                    FROM comp.fundq f
                    GROUP BY (f.gvkey, EXTRACT(YEAR FROM f.datadate))
                    ),
                    -- gvkey, 년도, 총이익, 총자산

                    fund_link AS(
                    SELECT 
                        f.gvkey,
                        f.year,
                        f.total_niq,
                        f.total_asset,
                        l.lpermno
                    FROM fund f
                    JOIN crsp.ccmxpf_linktable l
                    ON f.gvkey = l.gvkey
                    WHERE l.lpermno = 14593
                    AND l.linktype IN ('LU','LC')        -- Compustat↔CRSP 보편적 링크
                    AND l.linkprim IN ('P','C')          -- Primary / Company level
                    --ORDER BY f.year
                    --LIMIT 1
                    )
                    -- gvkey, 년도, 총이익, 총자산, permno

                    
                    -- 그러면 이제 div_y의 배당금, 배당률을 가지고 오면 됨
                    SELECT 
                        f.lpermno,
                        f.year,
                        d.total_div,
                        d.avg_div,
                        f.total_niq,
                        f.total_asset
                    FROM fund_link f
                    LEFT JOIN div_y d
                    ON d.year = f.year
                    AND d.permno = f.lpermno
                    ORDER BY f.year
                        """)

answer

,lpermno,year,total_div,avg_div,total_niq,total_asset
0,14593.0,1979.0,<NA>,<NA>,2.647,24.636
1,14593.0,1980.0,<NA>,<NA>,16.472,74.883
2,14593.0,1981.0,<NA>,<NA>,45.561,238.2775
3,14593.0,1982.0,<NA>,<NA>,71.263,348.0735
4,14593.0,1983.0,<NA>,<NA>,59.017,537.17725
5,14593.0,1984.0,<NA>,<NA>,104.332,772.396
6,14593.0,1985.0,<NA>,<NA>,72.049,926.78475
7,14593.0,1986.0,<NA>,<NA>,155.511,1151.69075
8,14593.0,1987.0,0.26,0.130459,280.391,1466.236
9,14593.0,1988.0,0.34,0.208972,419.342,1920.30675


: 

In [ ]:
apple_div = db.raw_sql("""
                    SELECT 
                        d.permno,
                        EXTRACT(YEAR FROM d.paydt) AS year,
                        SUM(d.divamt) AS total_div,
                        ABS(AVG(d.divamt/NULLIF(px.prc,0)))*100 AS avg_div
                    FROM crsp.dse d
                    LEFT JOIN LATERAL(
                        SELECT f.date, f.prc
                        FROM crsp.dsf f
                        WHERE d.permno = f.permno
                        AND f.date >= d.paydt
                        ORDER BY f.date
                        LIMIT 1) px ON TRUE
                    WHERE d.permno = 14593
                    GROUP BY (d.permno, EXTRACT(YEAR FROM d.paydt))
""")
apple_div

,permno,year,total_div,avg_div
0,14593,1987.0,0.26,0.225597
1,14593,1988.0,0.34,0.295011
2,14593,1989.0,0.41,0.355748
3,14593,1990.0,0.45,0.390456
4,14593,1991.0,0.48,0.416486
5,14593,1992.0,0.48,0.416486
6,14593,1993.0,0.48,0.416486
7,14593,1994.0,0.48,0.416486
8,14593,1995.0,0.48,0.416486
9,14593,2000.0,0.0,0.0
